# Nogai (nog) — Tokenisation and Morphological Analysis

Nogai (Kipchak branch) is supported via Cyrillic script tokenisation and Prototype-quality Apertium FST morphological analysis. NLLB-200 provides cross-lingual embeddings and machine translation.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('nog')

## 2. Tokenisation

In [ ]:
nlp_tok = Pipeline("nog", processors=["tokenize"])
doc = nlp_tok("Мен мектепке бараман.")
print([w.text for w in doc.words])

## 3. Script Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("NOGAI COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

cyrl = "Мен мектепке бараман."
print(f"Original (Cyrillic): {cyrl}")
print()

# Direction 1: Cyrillic → Turkic Common Alphabet (Latin)
print("1. Cyrillic → Turkic Common Alphabet (Latin):")
try:
    t1 = Transliterator("nog", source=Script.CYRILLIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(cyrl)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Cyrillic (reverse)
print("2. Turkic Common (Latin) → Cyrillic:")
try:
    t2 = Transliterator("nog", source=Script.COMMON_TURKIC, target=Script.CYRILLIC)
    back_to_cyrl = t2.transliterate(common if 'common' in locals() else "Men mektepke baraman.")
    print(f"   {back_to_cyrl}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Cyrillic → Latin
print("3. Cyrillic → Latin (explicit):")
try:
    t3 = Transliterator("nog", source=Script.CYRILLIC, target=Script.LATIN)
    latin = t3.transliterate(cyrl)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Cyrillic
print("4. Latin → Cyrillic:")
try:
    t4 = Transliterator("nog", source=Script.LATIN, target=Script.CYRILLIC)
    back_to_cyrl_explicit = t4.transliterate(latin if 'latin' in locals() else "Men mektepke baraman.")
    print(f"   {back_to_cyrl_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Nogai Scripts Supported:")
print("  • Cyrillic (primary)")
print("  • Latin (COMMON_TURKIC standard)")
print("=" * 70)

## 4. Morphological Analysis (Apertium FST — Prototype)

In [ ]:
nlp = Pipeline(
    "nog",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
)
doc = nlp("Мен мектепке бараман.")
for w in doc.words:
    print(f"{w.text:<18} lemma={w.lemma:<12} feats={w.feats}")

## 5. Translation via NLLB-200

In [ ]:
turkicnlp.download("nog", processors=["translate"])
trans = Pipeline("nog", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("Мен мектепке бараман.")
print("EN:", doc.translation)